# Modulo 2 — Clasificacion de Sentimiento (NLP)
**Objetivo:** Aplicar un modelo Transformer preentrenado en espanol para clasificar el sentimiento
de resenas del dataset de turismo mexicano. Salida: etiqueta {POS, NEG, NEU} y mapeo binario 0/1.

**Stack:** `pysentimiento` (RoBERTuito) | Python 3.14 | Dataset: MX Tourism Reviews

In [ ]:
import pandas as pd
import numpy as np
from datasets import load_dataset
from IPython.display import display

dataset = load_dataset("alexcom/analisis-sentimientos-textos-turisitcos-mx-polaridad")
df = dataset['train'].to_pandas()

print("Información del Dataset:")
df.info()

print("\nDistribución de Clases (Positivas vs Negativas):")
dist = df['label'].value_counts()
display(dist)


## Ingesta y Muestreo Estrategico
Carga del corpus completo y seleccion del 30% para inferencia representativa.
Se fija `random_state=42` para reproducibilidad entre ejecuciones.

In [ ]:
from transformers import pipeline
import torch

# Tomaremos una muestra del 30% para el enfoque de Transformers por eficiencia
df_sample_tf = df.sample(frac=0.3, random_state=42).copy()

# pysentimiento/robertuito-sentiment-analysis devuelve POS, NEG, NEU
device = 0 if torch.cuda.is_available() else -1
sentiment_pipeline = pipeline("sentiment-analysis", model="pysentimiento/robertuito-sentiment-analysis", device=device)

# Hacemos inferencia en lotes (batching) para no desbordar memoria
print(f"Realizando inferencia sobre {len(df_sample_tf)} registros...")
texts = df_sample_tf['text'].tolist()

# Inferencia real (puede tomar algo de tiempo dependiedo del equipo, truncamos a 512 tokens por texto si es necesario)
predictions = sentiment_pipeline(texts, truncation=True, max_length=128, batch_size=32)

# Extraer etiquetas predichas
df_sample_tf['pred_transformer'] = [p['label'] for p in predictions]
display(df_sample_tf.head())


## Inferencia con Modelo Transformer (RoBERTuito)
`RoBERTuito` es un modelo RoBERTa fine-tuned sobre 500M tokens de Twitter en espanol.
Se elige sobre enfoques basados en lexicon (VADER, AFINN) por su superioridad en textos
coloquiales y con errores ortograficos tipicos de resenas de usuarios.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Dividir 70/30 (Usaremos el 70% para train y evaluaremos en el 30% restante)
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['label'], test_size=0.3, random_state=42)

# Vectorización TF-IDF
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Entrenamiento de Logistic Regression
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_tfidf, y_train)

# Predicción
preds_lr = clf.predict(X_test_tfidf)


## Clasificacion con Enfoque TFIDF + Logistic Regression (Baseline)
Pipeline de referencia para contrastar contra el Transformer.
Su ventaja es velocidad de inferencia; su limitante es la ausencia de contexto semantico.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Alinear las predicciones del modelo A para poder compararlas con las etiquetas reales
# Mapeo: El modelo 'robertuito' suele devolver 'POS', 'NEG', 'NEU'. 
# Supongamos que nuestro dataset tiene 0 (Negativo) y 1 (Positivo).
label_map = {'POS': 1, 'NEG': 0, 'NEU': 0} # Agrupamos neutro con negativo o lo descartamos (depende de la convención)
df_sample_tf['pred_mapped'] = df_sample_tf['pred_transformer'].map(label_map).fillna(0)

print("=== Reporte de Clasificación (Enfoque A: Transformers) ===")
print(classification_report(df_sample_tf['label'], df_sample_tf['pred_mapped']))

print("\n=== Reporte de Clasificación (Enfoque B: TF-IDF + LogReg) ===")
print(classification_report(y_test, preds_lr))

# Matrices de Confusión
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_tf = confusion_matrix(df_sample_tf['label'], df_sample_tf['pred_mapped'])
disp_tf = ConfusionMatrixDisplay(confusion_matrix=cm_tf)
disp_tf.plot(ax=axes[0], cmap='Blues')
axes[0].set_title('Matriz de Confusión - Transformers')

cm_lr = confusion_matrix(y_test, preds_lr)
disp_lr = ConfusionMatrixDisplay(confusion_matrix=cm_lr)
disp_lr.plot(ax=axes[1], cmap='Greens')
axes[1].set_title('Matriz de Confusión - TF-IDF + LogReg')

plt.tight_layout()
plt.show()


## Consolidacion de Resultados y Mapeo Binario
Unificacion de predicciones bajo un esquema binario: POS=1, {NEG, NEU}=0.
Este mapeo facilita el calculo de metricas de precision operativa downstream.

## Persistencia — Artefacto de Salida
Exportacion del dataset clasificado a `data/modulo2_resultados.csv` para
consumo directo por el dashboard ejecutivo (Modulo 5).

In [ ]:
import os
if not os.path.exists('data'):
    os.makedirs('data')

# Exportamos los resultados del modelo Transformer (Enfoque A) al ser el más avanzado
export_df = df_sample_tf[['text', 'label', 'pred_transformer', 'pred_mapped']].copy()
export_df.rename(columns={'pred_transformer': 'prediccion_cruda', 'pred_mapped': 'prediccion_binaria'}, inplace=True)

export_df.to_csv('data/modulo2_resultados.csv', index=False)
print("Predicciones guardadas exitosamente en 'data/modulo2_resultados.csv'")
